# 전체 OD 경로 배정 (Route Assignment)
> OTP 경로 vs 스마트카드 통행 유사도 비교 → 경로 선택 확률 산출

**파이프라인**:
1. OTP 결과 JSON 로드 + 경로 중복 제거
2. TCN 데이터 로드 (전체 날짜)
3. OD별 유사도 계산 + 경로 배정
4. 결과 저장 + 요약 통계

---
## Part 1. OTP 결과 로드

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import glob
from datetime import datetime
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [ ]:
%load_ext autoreload
%autoreload 2

from module.similarity import (
    parse_otp_itinerary,
    parse_smartcard_trip,
    deduplicate_itineraries,
    compute_all_metrics,
    compute_composite_similarity,
    match_smartcard_to_otp,
    grade_similarity,
)

### 1-1. OTP 배치 JSON 로드

배치 JSON 형식: `{od_pair: [itinerary, ...], ...}`

`data/otp/output/` 디렉토리의 모든 JSON 파일을 로드합니다.

In [ ]:
OTP_OUTPUT_DIR = '../data/otp/output'

# 배치 JSON 파일 목록
json_files = sorted(glob.glob(os.path.join(OTP_OUTPUT_DIR, '*.json')))
print(f"JSON 파일 수: {len(json_files)}")
for f in json_files:
    print(f"  {os.path.basename(f)}")

In [ ]:
# 전체 OTP 결과 로드
otp_results_raw = {}  # od_pair → [itinerary, ...]

for fpath in tqdm(json_files, desc="JSON 로드"):
    with open(fpath, 'r', encoding='utf-8') as f:
        batch = json.load(f)
    
    for od_pair, itineraries in batch.items():
        if itineraries:  # 빈 결과 제외
            otp_results_raw[od_pair] = itineraries

print(f"\nOTP 결과 로드 완료")
print(f"  총 OD pairs: {len(otp_results_raw):,}")
print(f"  총 itineraries: {sum(len(v) for v in otp_results_raw.values()):,}")

### 1-2. 경로 중복 제거 (Deduplication)

출발시간만 다른 동일 경로를 합치고, 소요시간은 평균으로 대표합니다.

In [ ]:
otp_results = {}  # od_pair → deduplicated itineraries

total_before = 0
total_after = 0

for od_pair, itineraries in tqdm(otp_results_raw.items(), desc="중복 제거"):
    total_before += len(itineraries)
    deduped = deduplicate_itineraries(itineraries)
    otp_results[od_pair] = deduped
    total_after += len(deduped)

print(f"\n중복 제거 완료")
print(f"  전: {total_before:,}개 → 후: {total_after:,}개")
print(f"  OD당 평균 경로 수: {total_after / len(otp_results):.1f}개")
print(f"  경로 수 분포:")

route_counts = [len(v) for v in otp_results.values()]
for n in sorted(set(route_counts)):
    cnt = route_counts.count(n)
    print(f"    {n}개 경로: {cnt:,} ODs ({cnt/len(otp_results)*100:.1f}%)")

---
## Part 2. 유사도 매칭 + 경로 배정

### 2-1. TCN 데이터 로드

In [ ]:
TCN_DIR = '../data/tcn'
TCN_DATES = sorted(os.listdir(TCN_DIR))
print(f"TCN 날짜: {TCN_DATES}")

# 전체 날짜 TCN 로드 + 병합
tcn_list = []
for date in tqdm(TCN_DATES, desc="TCN 로드"):
    parquet_files = glob.glob(os.path.join(TCN_DIR, date, '*.parquet'))
    for pf in parquet_files:
        df = pd.read_parquet(pf)
        df['date'] = date
        tcn_list.append(df)
        print(f"  {date}: {len(df):,}건")

tcn = pd.concat(tcn_list, ignore_index=True)
print(f"\n전체 TCN: {len(tcn):,}건")
print(f"고유 OD pairs: {tcn['od_pair'].nunique():,}")

In [ ]:
# OTP 결과가 있는 OD만 필터링
otp_od_set = set(otp_results.keys())
tcn_filtered = tcn[tcn['od_pair'].isin(otp_od_set)].copy()

print(f"OTP 결과 존재하는 OD: {len(otp_od_set):,}")
print(f"TCN 중 매칭 대상: {len(tcn_filtered):,}건 ({len(tcn_filtered)/len(tcn)*100:.1f}%)")
print(f"매칭 대상 OD pairs: {tcn_filtered['od_pair'].nunique():,}")

# OD별로 그룹화 (메모리 효율)
tcn_grouped = dict(list(tcn_filtered.groupby('od_pair')))
print(f"\nOD 그룹 수: {len(tcn_grouped):,}")

### 2-2. 전체 유사도 계산 + 경로 배정

In [ ]:
RESULT_DIR = '../data/trip_assignment'
os.makedirs(RESULT_DIR, exist_ok=True)

# 이어하기 지원: 기존 결과가 있으면 건너뛰기
RESULT_PATH = os.path.join(RESULT_DIR, 'assignment_results.parquet')
CHECKPOINT_PATH = os.path.join(RESULT_DIR, 'assignment_checkpoint.parquet')
CHECKPOINT_INTERVAL = 5000  # 5000 OD마다 중간 저장

# 기존 체크포인트 로드
processed_ods = set()
prev_results = []
if os.path.exists(CHECKPOINT_PATH):
    prev_df = pd.read_parquet(CHECKPOINT_PATH)
    processed_ods = set(prev_df['od_pair'].unique())
    prev_results = prev_df.to_dict('records')
    print(f"체크포인트 로드: {len(processed_ods):,} OD 완료됨, {len(prev_results):,}건")
else:
    print("체크포인트 없음, 처음부터 시작")

In [ ]:
# 전체 OD 매칭 실행
all_results = list(prev_results)  # 이전 결과 이어받기
od_list = [od for od in tcn_grouped.keys() if od not in processed_ods]

print(f"처리 대상 OD: {len(od_list):,} (이미 완료: {len(processed_ods):,})")

for i, od_pair in enumerate(tqdm(od_list, desc="경로 배정")):
    itins = otp_results.get(od_pair, [])
    sc_trips = tcn_grouped[od_pair]
    
    if not itins or len(sc_trips) == 0:
        continue
    
    for _, sc_row in sc_trips.iterrows():
        sc_parsed = parse_smartcard_trip(sc_row)
        match_result = match_smartcard_to_otp(sc_parsed, itins)
        
        all_results.append({
            'od_pair': od_pair,
            'date': sc_row.get('date', ''),
            'sc_category': sc_row.get('transport_category', ''),
            'sc_transfers': sc_parsed['transfer_count'],
            'sc_time_sec': sc_parsed['total_time'],
            'matched': match_result['matched'],
            'best_score': round(match_result['best_score'], 4),
            'best_grade': match_result['best_grade'],
            'best_otp_idx': match_result['best_idx'],
            'mode_score': round(match_result['best_level_scores']['mode_score'], 4),
            'transfer_score': round(match_result['best_level_scores']['transfer_score'], 4),
            'sequence_score': round(match_result['best_level_scores']['sequence_score'], 4),
            'time_score': round(match_result['best_level_scores']['time_score'], 4),
            'route_score': round(match_result['best_level_scores']['route_score'], 4),
            'spatial_score': round(match_result['best_level_scores']['spatial_score'], 4),
        })
    
    # 중간 저장
    if (i + 1) % CHECKPOINT_INTERVAL == 0:
        ckpt_df = pd.DataFrame(all_results)
        ckpt_df.to_parquet(CHECKPOINT_PATH, index=False)
        print(f"  체크포인트 저장: {i+1:,}/{len(od_list):,} OD, {len(all_results):,}건")

# 최종 결과 저장
results_df = pd.DataFrame(all_results)
results_df.to_parquet(RESULT_PATH, index=False)

# 체크포인트 삭제
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)

print(f"\n완료!")
print(f"  총 매칭 건수: {len(results_df):,}")
print(f"  OD pairs: {results_df['od_pair'].nunique():,}")
print(f"  저장: {RESULT_PATH}")

---
## Part 3. 결과 요약

### 3-1. 전체 매칭 통계

In [ ]:
# 기본 통계
print("=== 전체 매칭 결과 ===")
print(f"총 SC 통행: {len(results_df):,}")
print(f"OD pairs: {results_df['od_pair'].nunique():,}")
print(f"매칭 성공률 (threshold=0.6): {results_df['matched'].mean()*100:.2f}%")
print(f"평균 복합점수: {results_df['best_score'].mean():.4f}")

print(f"\n=== 등급 분포 ===")
grade_dist = results_df['best_grade'].value_counts()
for grade in ['우수', '양호', '보통', '불량']:
    if grade in grade_dist.index:
        cnt = grade_dist[grade]
        print(f"  {grade}: {cnt:,} ({cnt/len(results_df)*100:.2f}%)")

print(f"\n=== 레벨별 평균 점수 ===")
level_cols = ['mode_score', 'transfer_score', 'sequence_score', 'time_score', 'route_score', 'spatial_score']
for col in level_cols:
    print(f"  {col}: {results_df[col].mean():.4f}")

print(f"\n=== 복합점수 통계 ===")
print(results_df['best_score'].describe())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 복합점수 분포
axes[0].hist(results_df['best_score'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
for th, color, ls in [(0.4, 'orange', ':'), (0.6, 'red', '-'), (0.8, 'green', '--')]:
    rate = (results_df['best_score'] >= th).mean() * 100
    axes[0].axvline(x=th, color=color, linestyle=ls, label=f'{th} ({rate:.1f}%)')
axes[0].set_xlabel('Composite Score')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Score Distribution (n={len(results_df):,})')
axes[0].legend(fontsize=9)

# 2. 레벨별 평균 점수
level_means = results_df[level_cols].mean()
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948']
axes[1].bar(range(len(level_cols)), level_means, color=colors,
            tick_label=[c.replace('_score', '') for c in level_cols])
axes[1].set_ylabel('Mean Score')
axes[1].set_title('Level-wise Mean Scores')
axes[1].set_ylim(0, 1.15)
for i, v in enumerate(level_means):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

# 3. 등급 분포 (파이차트)
grade_order = ['우수', '양호', '보통', '불량']
grade_colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
grade_counts = [grade_dist.get(g, 0) for g in grade_order]
nonzero = [(g, c, col) for g, c, col in zip(grade_order, grade_counts, grade_colors) if c > 0]
if nonzero:
    labels, counts, cols = zip(*nonzero)
    axes[2].pie(counts, labels=labels, colors=cols, autopct='%1.1f%%', startangle=90)
    axes[2].set_title('Grade Distribution')

plt.tight_layout()
plt.show()

### 3-2. OD별 경로 배정 확률

In [ ]:
# OD별 경로 배정 확률 계산
assignment_records = []

for od_pair, od_df in tqdm(results_df.groupby('od_pair'), desc="배정 확률 계산"):
    itins = otp_results.get(od_pair, [])
    total = len(od_df)
    matched = od_df[od_df['matched'] == True]
    unmatched_count = len(od_df[od_df['matched'] == False])
    
    for i, itin in enumerate(itins):
        otp_p = parse_otp_itinerary(itin)
        n = len(matched[matched['best_otp_idx'] == i])
        
        assignment_records.append({
            'od_pair': od_pair,
            'route_idx': i,
            'route_label': f'OTP 경로 {i}',
            'mode': ', '.join(sorted(otp_p['modes'])),
            'transport_category': otp_p['transport_category'],
            'transfers': otp_p['transfer_count'],
            'main_route': otp_p['main_route'],
            'time_min': round(otp_p['total_time'] / 60, 1),
            'stops': ' → '.join(otp_p['stops']),
            'assigned': n,
            'total_trips': total,
            'probability': round(n / total, 4),
        })
    
    # other (매칭 실패)
    assignment_records.append({
        'od_pair': od_pair,
        'route_idx': -1,
        'route_label': 'other',
        'mode': '-',
        'transport_category': '-',
        'transfers': -1,
        'main_route': '-',
        'time_min': -1,
        'stops': '-',
        'assigned': unmatched_count,
        'total_trips': total,
        'probability': round(unmatched_count / total, 4),
    })

assignment_df = pd.DataFrame(assignment_records)

# 저장
ASSIGNMENT_PATH = os.path.join(RESULT_DIR, 'route_assignment_probabilities.parquet')
assignment_df.to_parquet(ASSIGNMENT_PATH, index=False)

print(f"경로 배정 확률 계산 완료")
print(f"  OD pairs: {assignment_df['od_pair'].nunique():,}")
print(f"  총 레코드: {len(assignment_df):,}")
print(f"  저장: {ASSIGNMENT_PATH}")

In [ ]:
# 배정 결과 요약 통계
# OD별: 1위 경로의 확률 분포
top1_probs = assignment_df[assignment_df['route_idx'] >= 0].groupby('od_pair')['probability'].max()
other_probs = assignment_df[assignment_df['route_idx'] == -1].set_index('od_pair')['probability']

print("=== 배정 결과 요약 ===")
print(f"\n1위 경로 확률 분포:")
print(top1_probs.describe())

print(f"\nother (매칭 실패) 확률 분포:")
print(other_probs.describe())

# 확률 > 0인 경로 수 분포
active_routes = assignment_df[(assignment_df['route_idx'] >= 0) & (assignment_df['assigned'] > 0)]
routes_per_od = active_routes.groupby('od_pair').size()
print(f"\nOD당 활성 경로 수 분포:")
print(routes_per_od.value_counts().sort_index())

In [ ]:
# 시각화: 1위 경로 확률 분포 + other 비율
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(top1_probs, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Top-1 Route Probability')
axes[0].set_ylabel('OD Count')
axes[0].set_title(f'Top-1 Route Probability Distribution (n={len(top1_probs):,} ODs)')
axes[0].axvline(x=top1_probs.mean(), color='red', linestyle='--', label=f'Mean={top1_probs.mean():.3f}')
axes[0].legend()

axes[1].hist(other_probs, bins=50, edgecolor='black', alpha=0.7, color='salmon')
axes[1].set_xlabel('Other (Unmatched) Probability')
axes[1].set_ylabel('OD Count')
axes[1].set_title(f'Unmatched Rate Distribution')
axes[1].axvline(x=other_probs.mean(), color='red', linestyle='--', label=f'Mean={other_probs.mean():.3f}')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3-3. transport_category별 통계

In [ ]:
# SC transport_category별 매칭 성공률 + 평균 점수
cat_stats = results_df.groupby('sc_category').agg(
    count=('best_score', 'count'),
    match_rate=('matched', 'mean'),
    mean_score=('best_score', 'mean'),
    mean_mode=('mode_score', 'mean'),
    mean_transfer=('transfer_score', 'mean'),
    mean_sequence=('sequence_score', 'mean'),
    mean_time=('time_score', 'mean'),
    mean_route=('route_score', 'mean'),
    mean_spatial=('spatial_score', 'mean'),
).sort_values('count', ascending=False)

cat_stats['match_rate'] = (cat_stats['match_rate'] * 100).round(2)

print("=== transport_category별 매칭 통계 ===")
print(cat_stats.to_string())

### 3-4. 샘플 OD 상세 확인

In [ ]:
# 통행량 상위 20개 OD 배정 결과
top_ods = results_df.groupby('od_pair').size().nlargest(20).index.tolist()

for od in top_ods[:5]:  # 상위 5개만 출력
    sub = assignment_df[assignment_df['od_pair'] == od]
    total = sub['total_trips'].iloc[0]
    print(f"\n--- {od} (총 {total:,}건) ---")
    display_cols = ['route_label', 'mode', 'transfers', 'main_route', 'time_min', 'assigned', 'probability']
    print(sub[display_cols].to_string(index=False))

---
## Part 4. 최종 저장

In [ ]:
# CSV 저장 (외부 활용용)
assignment_df.to_csv(os.path.join(RESULT_DIR, 'route_assignment_probabilities.csv'), 
                     index=False, encoding='utf-8-sig')

# OTP 결과도 저장 (재사용용)
with open(os.path.join(RESULT_DIR, 'otp_results_deduped.json'), 'w', encoding='utf-8') as f:
    json.dump(otp_results, f, ensure_ascii=False)

print("=== 저장 완료 ===")
print(f"  경로 배정 결과: {RESULT_DIR}/route_assignment_probabilities.parquet")
print(f"  경로 배정 결과 (CSV): {RESULT_DIR}/route_assignment_probabilities.csv")
print(f"  전체 매칭 상세: {RESULT_DIR}/assignment_results.parquet")
print(f"  OTP 중복제거 결과: {RESULT_DIR}/otp_results_deduped.json")